# Model Selection Training

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 79.


In [ ]:
# ============================================================
# MODEL-SELECTION / CHECKPOINT-SELECTION SENSITIVITY
# Morgan + RDKit | Seeds 42, 43, 44
# ============================================================
#
# CONTROLLED FACTOR:
#   checkpoint selection criterion ONLY
#
# Criteria saved from CALIBRATION:
#   1) best Macro-F1
#   2) best Accuracy
#   3) best Weighted-F1
#
# Fixed:
# - Morgan + RDKit Train-only features = 2249
# - same Train / Calibration / Test splits
# - same 84 classes
# - same ordered Drug1 -> Drug2 representation
# - same graph construction
# - same GCN architecture
# - same weighted CE
# - same optimizer / scheduler
# - same Macro-F1-driven early stopping
# - Test evaluated ONLY after training/model selection
#
# ============================================================

import os
import gc
import json
import math
import random
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    f1_score,
)

from torch_geometric.nn import GCNConv


# ============================================================
# 0) CONFIG
# ============================================================

SEEDS = [42, 43, 44]

NUM_CLASSES = 84

BATCH_SIZE = 512
MAX_EPOCHS = 120
EARLY_STOP_PATIENCE = 18

LR = 7e-4
WEIGHT_DECAY = 1e-4

DROPOUT = 0.35

ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/ddi-v2-menna-projectt"
)

FEATURE_ROOT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING"
)

FEATURE_FILE = (
    FEATURE_ROOT
    /
    "molecular_features_morgan_rdkit.npz"
)

OUT_ROOT = Path(
    "/kaggle/working/"
    "MODEL_SELECTION_SENSITIVITY_MORGAN_RDKIT_V1"
)

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


TRAIN_FILE = ROOT / "clean_train_split.csv"
CAL_FILE   = ROOT / "clean_calibration_split.csv"
TEST_FILE  = ROOT / "clean_test_split.csv"


# ============================================================
# 1) HELPERS
# ============================================================

def sha256_file(path, block_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(block_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_rng_state():

    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }

    if torch.cuda.is_available():

        state["cuda"] = (
            torch.cuda.get_rng_state_all()
        )

    else:

        state["cuda"] = None

    return state


def set_rng_state(state):

    random.setstate(
        state["python"]
    )

    np.random.set_state(
        state["numpy"]
    )

    torch.set_rng_state(
        state["torch"]
    )

    if (
        torch.cuda.is_available()
        and
        state["cuda"] is not None
    ):

        torch.cuda.set_rng_state_all(
            state["cuda"]
        )


# ============================================================
# 2) LOAD EXACT DATA
# ============================================================

assert FEATURE_FILE.is_file()
assert TRAIN_FILE.is_file()
assert CAL_FILE.is_file()
assert TEST_FILE.is_file()


train_df = pd.read_csv(TRAIN_FILE)
cal_df   = pd.read_csv(CAL_FILE)
test_df  = pd.read_csv(TEST_FILE)


assert len(train_df) == 134249
assert len(cal_df) == 28768
assert len(test_df) == 28768


print("=" * 90)
print("DATA SPLITS")
print("=" * 90)

print("Train:", train_df.shape)
print("Calibration:", cal_df.shape)
print("Test:", test_df.shape)


# ============================================================
# 3) VERIFY IMMUTABLE INTERACTION LABEL MAPPING
# ============================================================

required_cols = {
    "Drug1",
    "Drug2",
    "Label",
    "label_id",
}

for name, df in [
    ("Train", train_df),
    ("Calibration", cal_df),
    ("Test", test_df),
]:

    missing = required_cols - set(df.columns)

    assert not missing, (
        f"{name} missing columns: {missing}"
    )


mapping_df = pd.concat(
    [
        train_df[["label_id", "Label"]],
        cal_df[["label_id", "Label"]],
        test_df[["label_id", "Label"]],
    ],
    ignore_index=True,
).drop_duplicates()


# one ID -> one exact Label
assert (
    mapping_df.groupby("label_id")["Label"]
    .nunique()
    .max()
    ==
    1
)

# one exact Label -> one ID
assert (
    mapping_df.groupby("Label")["label_id"]
    .nunique()
    .max()
    ==
    1
)

assert mapping_df["label_id"].nunique() == 84
assert mapping_df["Label"].nunique() == 84


# robust internal 0..83 indexing
label_ids_sorted = sorted(
    mapping_df["label_id"].unique().tolist()
)

label_to_internal = {
    original_id: i
    for i, original_id in enumerate(label_ids_sorted)
}


def encode_y(df):

    y = (
        df["label_id"]
        .map(label_to_internal)
        .to_numpy(dtype=np.int64)
    )

    assert not np.isnan(y).any()

    return y


y_train = encode_y(train_df)
y_cal   = encode_y(cal_df)
y_test  = encode_y(test_df)


assert len(np.unique(y_train)) == 84
assert len(np.unique(y_cal)) == 84
assert len(np.unique(y_test)) == 84


print(
    " 84 Interaction IDs/labels verified unchanged."
)


# ============================================================
# 4) LOAD TRAIN-ONLY MORGAN + RDKIT FEATURES
# ============================================================

with np.load(
    FEATURE_FILE,
    allow_pickle=True,
) as data:

    X = data["x"].astype(
        np.float32
    )

    all_drugs = (
        data["all_drugs"]
        .astype(str)
    )


assert X.shape == (
    1705,
    2249,
)

assert len(all_drugs) == 1705
assert len(set(all_drugs.tolist())) == 1705
assert np.isfinite(X).all()


drug_to_global = {
    drug: i
    for i, drug in enumerate(all_drugs)
}


for df_name, df in [
    ("Train", train_df),
    ("Calibration", cal_df),
    ("Test", test_df),
]:

    for column in [
        "Drug1",
        "Drug2",
    ]:

        missing = (
            set(df[column].astype(str))
            -
            set(drug_to_global)
        )

        assert not missing, (
            f"{df_name}/{column}: "
            f"{len(missing)} drugs absent from feature cache"
        )


print(
    " Morgan + RDKit feature cache:",
    X.shape
)


# ============================================================
# 5) BUILD ORDERED PAIR ARRAYS
# ============================================================

def make_pairs(df):

    d1 = (
        df["Drug1"]
        .astype(str)
        .map(drug_to_global)
        .to_numpy(dtype=np.int64)
    )

    d2 = (
        df["Drug2"]
        .astype(str)
        .map(drug_to_global)
        .to_numpy(dtype=np.int64)
    )

    return np.column_stack(
        [d1, d2]
    )


train_pairs_global = make_pairs(
    train_df
)

cal_pairs_global = make_pairs(
    cal_df
)

test_pairs_global = make_pairs(
    test_df
)


assert train_pairs_global.shape == (
    134249,
    2,
)


# ============================================================
# 6) EXACT GRAPH RECIPE
# ============================================================
#
# IMPORTANT:
# 1. unique ORIGINAL ordered Train pairs
# 2. reverse each unique pair
# 3. stack
# 4. NO SECOND GLOBAL DEDUP
#
# ============================================================

forward_unique = np.unique(
    train_pairs_global,
    axis=0,
)

assert forward_unique.shape == (
    134122,
    2,
)


reverse_unique = (
    forward_unique[:, ::-1]
    .copy()
)


forward_set = {
    (int(a), int(b))
    for a, b in forward_unique
}

reverse_set = {
    (int(a), int(b))
    for a, b in reverse_unique
}


reciprocal_overlap = len(
    forward_set.intersection(
        reverse_set
    )
)

assert reciprocal_overlap == 122


diagnostic_global_unique = np.unique(
    np.vstack(
        [
            forward_unique,
            reverse_unique,
        ]
    ),
    axis=0,
)

assert diagnostic_global_unique.shape == (
    268122,
    2,
)


graph_pairs_global = np.vstack(
    [
        forward_unique,
        reverse_unique,
    ]
)

assert graph_pairs_global.shape == (
    268244,
    2,
)


print("\n" + "=" * 90)
print("GRAPH QA")
print("=" * 90)

print(
    "Raw Train rows:",
    len(train_pairs_global)
)

print(
    "Unique ordered Train edges:",
    len(forward_unique)
)

print(
    "Reciprocal directed overlap:",
    reciprocal_overlap
)

print(
    "Diagnostic global unique:",
    len(diagnostic_global_unique)
)

print(
    "ACTUAL graph edges:",
    len(graph_pairs_global)
)

print(
    " No second global deduplication."
)


# ============================================================
# 7) TRAIN-ONLY LOCAL GRAPH
# ============================================================

train_drug_global = np.unique(
    train_pairs_global.reshape(-1)
)

assert len(train_drug_global) == 1675


global_to_local = {
    int(global_idx): local_idx
    for local_idx, global_idx
    in enumerate(train_drug_global)
}


train_pairs_local = np.array(
    [
        [
            global_to_local[int(a)],
            global_to_local[int(b)],
        ]
        for a, b in train_pairs_global
    ],
    dtype=np.int64,
)


graph_pairs_local = np.array(
    [
        [
            global_to_local[int(a)],
            global_to_local[int(b)],
        ]
        for a, b in graph_pairs_global
    ],
    dtype=np.int64,
)


X_train_local = X[
    train_drug_global
]


assert X_train_local.shape == (
    1675,
    2249,
)


# held-out graph leakage QA
graph_global_nodes = set(
    graph_pairs_global.reshape(-1).tolist()
)

train_global_nodes = set(
    train_drug_global.tolist()
)

assert (
    graph_global_nodes
    ==
    train_global_nodes
)


cal_nodes = set(
    cal_pairs_global.reshape(-1).tolist()
)

test_nodes = set(
    test_pairs_global.reshape(-1).tolist()
)


cal_only = (
    cal_nodes
    -
    train_global_nodes
)

test_only = (
    test_nodes
    -
    train_global_nodes
)


assert len(cal_only) == 14
assert len(test_only) == 20

assert not (
    cal_only
    &
    graph_global_nodes
)

assert not (
    test_only
    &
    graph_global_nodes
)


print(
    "Train graph drugs:",
    len(train_global_nodes)
)

print(
    "Calibration unseen drugs:",
    len(cal_only)
)

print(
    "Test unseen drugs:",
    len(test_only)
)

print(
    " Held-out-only drugs are not incident "
    "to Train interaction edges."
)


# ============================================================
# 8) TORCH TENSORS
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


print(
    "\nDevice:",
    device
)


x_all = torch.tensor(
    X,
    dtype=torch.float32,
    device=device,
)

x_train = torch.tensor(
    X_train_local,
    dtype=torch.float32,
    device=device,
)


edge_train = torch.tensor(
    graph_pairs_local.T,
    dtype=torch.long,
    device=device,
)

edge_global = torch.tensor(
    graph_pairs_global.T,
    dtype=torch.long,
    device=device,
)


train_pairs_t = torch.tensor(
    train_pairs_local,
    dtype=torch.long,
    device=device,
)

cal_pairs_t = torch.tensor(
    cal_pairs_global,
    dtype=torch.long,
    device=device,
)

test_pairs_t = torch.tensor(
    test_pairs_global,
    dtype=torch.long,
    device=device,
)


y_train_t = torch.tensor(
    y_train,
    dtype=torch.long,
    device=device,
)


# ============================================================
# 9) CLASS WEIGHTS
# ============================================================

counts = np.bincount(
    y_train,
    minlength=NUM_CLASSES,
).astype(np.float64)


assert np.all(
    counts > 0
)


max_count = counts.max()


class_weights = np.sqrt(
    max_count
    /
    (
        counts + 1.0
    )
)


class_weights = np.clip(
    class_weights,
    0.5,
    3.0,
)


class_weights = (
    class_weights
    /
    class_weights.mean()
)


class_weights_t = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device,
)


print(
    "\nClass weight range:",
    float(class_weights.min()),
    "to",
    float(class_weights.max())
)


# ============================================================
# 10) MODEL
# ============================================================

class GCNDrugEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        dropout=0.35,
    ):

        super().__init__()

        self.conv1 = GCNConv(
            input_dim,
            384,
            cached=False,
            add_self_loops=True,
            normalize=True,
        )

        self.bn1 = nn.BatchNorm1d(
            384
        )

        self.conv2 = GCNConv(
            384,
            384,
            cached=False,
            add_self_loops=True,
            normalize=True,
        )

        self.bn2 = nn.BatchNorm1d(
            384
        )

        self.conv3 = GCNConv(
            384,
            192,
            cached=False,
            add_self_loops=True,
            normalize=True,
        )

        self.bn3 = nn.BatchNorm1d(
            192
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        edge_index,
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = self.bn1(x)

        x = F.relu(x)

        x = self.dropout(x)


        x = self.conv2(
            x,
            edge_index
        )

        x = self.bn2(x)

        x = F.relu(x)

        x = self.dropout(x)


        x = self.conv3(
            x,
            edge_index
        )

        x = self.bn3(x)

        # intentionally no final activation
        return x


class FinalConcatOnlyGCN(nn.Module):

    def __init__(
        self,
        input_dim,
        num_classes=84,
        dropout=0.35,
    ):

        super().__init__()

        self.encoder = GCNDrugEncoder(
            input_dim=input_dim,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(

            nn.LayerNorm(
                384
            ),

            nn.Linear(
                384,
                512
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                512,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                num_classes
            ),
        )


    def classify_pairs(
        self,
        z,
        pair_idx,
    ):

        z1 = z[
            pair_idx[:, 0]
        ]

        z2 = z[
            pair_idx[:, 1]
        ]

        ordered_concat_z1_z2 = (
            torch.cat(
                [z1, z2],
                dim=1,
            )
        )

        return self.classifier(
            ordered_concat_z1_z2
        )


# ============================================================
# 11) METRIC FUNCTIONS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
):

    return {

        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    y_pred,
                )
            ),

        "macro84":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    labels=np.arange(
                        NUM_CLASSES
                    ),
                    average="macro",
                    zero_division=0,
                )
            ),

        "weighted_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    labels=np.arange(
                        NUM_CLASSES
                    ),
                    average="weighted",
                    zero_division=0,
                )
            ),
    }


@torch.no_grad()
def evaluate_split(
    model,
    x,
    edge_index,
    pair_tensor,
    y_true,
):

    model.eval()

    z = model.encoder(
        x,
        edge_index,
    )


    pred_chunks = []


    for start in range(
        0,
        len(pair_tensor),
        BATCH_SIZE,
    ):

        end = min(
            start + BATCH_SIZE,
            len(pair_tensor),
        )

        logits = model.classify_pairs(
            z,
            pair_tensor[
                start:end
            ],
        )

        preds = (
            logits.argmax(
                dim=1
            )
            .detach()
            .cpu()
            .numpy()
        )

        pred_chunks.append(
            preds
        )


    y_pred = np.concatenate(
        pred_chunks
    )


    metrics = calculate_metrics(
        y_true,
        y_pred,
    )


    return (
        metrics,
        y_pred,
    )


# ============================================================
# 12) CHECKPOINT SAVE
# ============================================================

def save_model_checkpoint(
    path,
    model,
    seed,
    epoch,
    criterion_name,
    cal_metrics,
):

    payload = {

        "seed":
            int(seed),

        "epoch":
            int(epoch),

        "criterion":
            criterion_name,

        "cal_metrics":
            cal_metrics,

        "model_state":
            {
                k:
                    v.detach()
                    .cpu()
                    .clone()

                for k, v
                in model.state_dict().items()
            },
    }


    torch.save(
        payload,
        path,
    )


# ============================================================
# 13) TRAIN ONE SEED
# ============================================================

def run_seed(seed):

    print(
        "\n"
        +
        "#" * 90
    )

    print(
        f"MODEL-SELECTION SENSITIVITY — SEED {seed}"
    )

    print(
        "#" * 90
    )


    seed_dir = (
        OUT_ROOT
        /
        f"seed_{seed}"
    )

    seed_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    final_json = (
        seed_dir
        /
        "final_result.json"
    )


    # --------------------------------------------------------
    # If fully completed, read instead of retraining
    # --------------------------------------------------------

    if final_json.is_file():

        print(
            f" Seed {seed} already completed."
        )

        return json.loads(
            final_json.read_text(
                encoding="utf-8"
            )
        )


    set_seed(seed)


    model = FinalConcatOnlyGCN(
        input_dim=2249,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT,
    ).to(device)


    total_params = sum(
        p.numel()
        for p in model.parameters()
    )


    criterion = nn.CrossEntropyLoss(
        weight=class_weights_t
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )


    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,
        )
    )


    best_paths = {

        "macro_f1":
            seed_dir
            /
            "best_by_macro_f1.pt",

        "accuracy":
            seed_dir
            /
            "best_by_accuracy.pt",

        "weighted_f1":
            seed_dir
            /
            "best_by_weighted_f1.pt",
    }


    last_path = (
        seed_dir
        /
        "last_training_state.pt"
    )


    best_values = {

        "macro_f1":
            -np.inf,

        "accuracy":
            -np.inf,

        "weighted_f1":
            -np.inf,
    }


    best_epochs = {

        "macro_f1":
            None,

        "accuracy":
            None,

        "weighted_f1":
            None,
    }


    start_epoch = 1
    epochs_without_macro_improvement = 0


    history = []


    # --------------------------------------------------------
    # RESUME
    # --------------------------------------------------------

    if last_path.is_file():

        state = torch.load(
            last_path,
            map_location="cpu",
            weights_only=False,
        )


        model.load_state_dict(
            state["model_state"]
        )


        optimizer.load_state_dict(
            state["optimizer_state"]
        )


        # move optimizer tensors to current device
        for opt_state in optimizer.state.values():

            for key, value in opt_state.items():

                if torch.is_tensor(value):

                    opt_state[key] = value.to(
                        device
                    )


        scheduler.load_state_dict(
            state["scheduler_state"]
        )


        best_values = state[
            "best_values"
        ]

        best_epochs = state[
            "best_epochs"
        ]

        history = state[
            "history"
        ]

        epochs_without_macro_improvement = (
            state[
                "epochs_without_macro_improvement"
            ]
        )


        start_epoch = (
            int(
                state["epoch"]
            )
            +
            1
        )


        set_rng_state(
            state["rng_state"]
        )


        print(
            f"↻ Resuming Seed {seed} "
            f"from epoch {start_epoch}"
        )


    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    for epoch in range(
        start_epoch,
        MAX_EPOCHS + 1,
    ):

        model.train()


        permutation = torch.randperm(
            len(train_pairs_t),
            device=device,
        )


        total_loss = 0.0
        total_seen = 0


        for start in range(
            0,
            len(train_pairs_t),
            BATCH_SIZE,
        ):

            idx = permutation[
                start:
                start + BATCH_SIZE
            ]


            optimizer.zero_grad(
                set_to_none=True
            )


            z = model.encoder(
                x_train,
                edge_train,
            )


            logits = model.classify_pairs(
                z,
                train_pairs_t[idx],
            )


            loss = criterion(
                logits,
                y_train_t[idx],
            )


            loss.backward()


            optimizer.step()


            batch_n = len(idx)


            total_loss += (
                float(
                    loss.detach()
                    .cpu()
                    .item()
                )
                *
                batch_n
            )


            total_seen += (
                batch_n
            )


        train_loss = (
            total_loss
            /
            total_seen
        )


        # ----------------------------------------------------
        # CALIBRATION ONLY
        # ----------------------------------------------------

        cal_metrics, _ = evaluate_split(
            model=model,
            x=x_all,
            edge_index=edge_global,
            pair_tensor=cal_pairs_t,
            y_true=y_cal,
        )


        cal_macro = (
            cal_metrics[
                "macro84"
            ]
        )

        cal_acc = (
            cal_metrics[
                "accuracy"
            ]
        )

        cal_weighted = (
            cal_metrics[
                "weighted_f1"
            ]
        )


        scheduler.step(
            cal_macro
        )


        lr_now = optimizer.param_groups[
            0
        ]["lr"]


        # ----------------------------------------------------
        # SAVE BEST BY MACRO-F1
        # ----------------------------------------------------

        macro_improved = (
            cal_macro
            >
            best_values[
                "macro_f1"
            ]
        )


        if macro_improved:

            best_values[
                "macro_f1"
            ] = (
                cal_macro
            )

            best_epochs[
                "macro_f1"
            ] = (
                epoch
            )


            save_model_checkpoint(
                path=best_paths[
                    "macro_f1"
                ],
                model=model,
                seed=seed,
                epoch=epoch,
                criterion_name=(
                    "calibration_macro_f1"
                ),
                cal_metrics=cal_metrics,
            )


            epochs_without_macro_improvement = 0


        else:

            epochs_without_macro_improvement += 1


        # ----------------------------------------------------
        # SAVE BEST BY ACCURACY
        # ----------------------------------------------------

        if (
            cal_acc
            >
            best_values[
                "accuracy"
            ]
        ):

            best_values[
                "accuracy"
            ] = (
                cal_acc
            )

            best_epochs[
                "accuracy"
            ] = (
                epoch
            )


            save_model_checkpoint(
                path=best_paths[
                    "accuracy"
                ],
                model=model,
                seed=seed,
                epoch=epoch,
                criterion_name=(
                    "calibration_accuracy"
                ),
                cal_metrics=cal_metrics,
            )


        # ----------------------------------------------------
        # SAVE BEST BY WEIGHTED-F1
        # ----------------------------------------------------

        if (
            cal_weighted
            >
            best_values[
                "weighted_f1"
            ]
        ):

            best_values[
                "weighted_f1"
            ] = (
                cal_weighted
            )

            best_epochs[
                "weighted_f1"
            ] = (
                epoch
            )


            save_model_checkpoint(
                path=best_paths[
                    "weighted_f1"
                ],
                model=model,
                seed=seed,
                epoch=epoch,
                criterion_name=(
                    "calibration_weighted_f1"
                ),
                cal_metrics=cal_metrics,
            )


        row = {

            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "lr":
                lr_now,

            "cal_accuracy":
                cal_acc,

            "cal_macro84":
                cal_macro,

            "cal_weighted_f1":
                cal_weighted,

            "best_macro_epoch":
                best_epochs[
                    "macro_f1"
                ],

            "best_accuracy_epoch":
                best_epochs[
                    "accuracy"
                ],

            "best_weighted_epoch":
                best_epochs[
                    "weighted_f1"
                ],
        }


        history.append(
            row
        )


        print(
            f"Seed {seed} | "
            f"Epoch {epoch:03d} | "
            f"Loss {train_loss:.6f} | "
            f"CAL Acc {cal_acc:.6f} | "
            f"Macro84 {cal_macro:.6f} | "
            f"Weighted {cal_weighted:.6f} | "
            f"LR {lr_now:.2e}"
        )


        # ----------------------------------------------------
        # SAVE RESUME STATE AFTER EVERY EPOCH
        # ----------------------------------------------------

        resume_payload = {

            "seed":
                seed,

            "epoch":
                epoch,

            "model_state":
                model.state_dict(),

            "optimizer_state":
                optimizer.state_dict(),

            "scheduler_state":
                scheduler.state_dict(),

            "best_values":
                best_values,

            "best_epochs":
                best_epochs,

            "epochs_without_macro_improvement":
                epochs_without_macro_improvement,

            "history":
                history,

            "rng_state":
                get_rng_state(),
        }


        torch.save(
            resume_payload,
            last_path,
        )


        pd.DataFrame(
            history
        ).to_csv(
            seed_dir
            /
            "training_history.csv",
            index=False,
        )


        # ----------------------------------------------------
        # BASELINE EARLY STOP RULE = Macro-F1
        # ----------------------------------------------------

        if (
            epochs_without_macro_improvement
            >=
            EARLY_STOP_PATIENCE
        ):

            print(
                f"\nEarly stopping at epoch {epoch} "
                f"based on Calibration Macro-F1."
            )

            break


    completed_epochs = (
        history[-1]["epoch"]
    )


    # ========================================================
    # TEST IS TOUCHED ONLY NOW
    # ========================================================

    print(
        "\n"
        +
        "=" * 90
    )

    print(
        "FINAL TEST EVALUATION"
    )

    print(
        "=" * 90
    )

    print(
        "Training/model selection finished."
    )

    print(
        "Now evaluating the three "
        "Calibration-selected checkpoints on TEST."
    )


    criterion_results = {}


    checkpoint_specs = [

        (
            "macro_f1",
            "Calibration Macro-F1",
        ),

        (
            "accuracy",
            "Calibration Accuracy",
        ),

        (
            "weighted_f1",
            "Calibration Weighted-F1",
        ),
    ]


    for key, display_name in checkpoint_specs:

        path = best_paths[
            key
        ]


        assert path.is_file()


        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )


        model.load_state_dict(
            checkpoint[
                "model_state"
            ]
        )


        model.to(
            device
        )


        # Recalculate CAL for checkpoint QA
        reloaded_cal, _ = evaluate_split(
            model=model,
            x=x_all,
            edge_index=edge_global,
            pair_tensor=cal_pairs_t,
            y_true=y_cal,
        )


        # Evaluate TEST only now
        test_metrics, test_pred = evaluate_split(
            model=model,
            x=x_all,
            edge_index=edge_global,
            pair_tensor=test_pairs_t,
            y_true=y_test,
        )


        pred_path = (
            seed_dir
            /
            f"test_predictions_best_by_{key}.npy"
        )


        np.save(
            pred_path,
            test_pred,
        )


        criterion_results[
            key
        ] = {

            "selection_name":
                display_name,

            "selected_epoch":
                int(
                    checkpoint[
                        "epoch"
                    ]
                ),

            "saved_cal_accuracy":
                float(
                    checkpoint[
                        "cal_metrics"
                    ][
                        "accuracy"
                    ]
                ),

            "saved_cal_macro84":
                float(
                    checkpoint[
                        "cal_metrics"
                    ][
                        "macro84"
                    ]
                ),

            "saved_cal_weighted_f1":
                float(
                    checkpoint[
                        "cal_metrics"
                    ][
                        "weighted_f1"
                    ]
                ),

            "reloaded_cal_accuracy":
                reloaded_cal[
                    "accuracy"
                ],

            "reloaded_cal_macro84":
                reloaded_cal[
                    "macro84"
                ],

            "reloaded_cal_weighted_f1":
                reloaded_cal[
                    "weighted_f1"
                ],

            "test_accuracy":
                test_metrics[
                    "accuracy"
                ],

            "test_macro84":
                test_metrics[
                    "macro84"
                ],

            "test_weighted_f1":
                test_metrics[
                    "weighted_f1"
                ],

            "checkpoint_sha256":
                sha256_file(
                    path
                ),

            "test_predictions_sha256":
                sha256_file(
                    pred_path
                ),
        }


        print(
            "\n",
            display_name
        )

        print(
            "Selected epoch:",
            checkpoint["epoch"]
        )

        print(
            "CAL Acc:",
            reloaded_cal[
                "accuracy"
            ]
        )

        print(
            "CAL Macro84:",
            reloaded_cal[
                "macro84"
            ]
        )

        print(
            "CAL Weighted:",
            reloaded_cal[
                "weighted_f1"
            ]
        )

        print(
            "TEST Acc:",
            test_metrics[
                "accuracy"
            ]
        )

        print(
            "TEST Macro84:",
            test_metrics[
                "macro84"
            ]
        )

        print(
            "TEST Weighted:",
            test_metrics[
                "weighted_f1"
            ]
        )


    # ========================================================
    # SEED RESULT
    # ========================================================

    result = {

        "experiment":
            "checkpoint_selection_sensitivity",

        "feature_mode":
            "morgan_rdkit",

        "seed":
            seed,

        "input_dim":
            2249,

        "num_classes":
            84,

        "model":
            "concat_only_gcn",

        "graph":
            "TRAIN_ORDERED_UNIQUE_THEN_REVERSE",

        "graph_forward_unique_edges":
            134122,

        "graph_total_edges":
            268244,

        "graph_reciprocal_overlap":
            122,

        "train_rows":
            134249,

        "calibration_rows":
            28768,

        "test_rows":
            28768,

        "train_fit_drugs":
            1675,

        "total_drugs":
            1705,

        "calibration_unseen_drugs":
            14,

        "test_unseen_drugs":
            20,

        "training_early_stopping_criterion":
            "calibration_macro_f1",

        "scheduler_criterion":
            "calibration_macro_f1",

        "early_stopping_patience":
            EARLY_STOP_PATIENCE,

        "max_epochs":
            MAX_EPOCHS,

        "completed_epochs":
            completed_epochs,

        "total_parameters":
            total_params,

        "selection_results":
            criterion_results,

        "feature_cache_sha256":
            sha256_file(
                FEATURE_FILE
            ),
    }


    final_json.write_text(
        json.dumps(
            result,
            indent=2,
        ),
        encoding="utf-8",
    )


    # ========================================================
    # SEED ZIP
    # ========================================================

    zip_path = (
        seed_dir
        /
        (
            f"MODEL_SELECTION_SENSITIVITY_"
            f"SEED_{seed}_COMPLETE.zip"
        )
    )


    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as z:

        for file_path in seed_dir.iterdir():

            if (
                file_path.is_file()
                and
                file_path != zip_path
            ):

                z.write(
                    file_path,
                    arcname=file_path.name,
                )


    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as z:

        assert z.testzip() is None


    print(
        f"\n Seed {seed} complete."
    )

    print(
        "ZIP:",
        zip_path
    )


    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    return result


# ============================================================
# 14) RUN 42 / 43 / 44
# ============================================================

all_seed_results = []


for seed in SEEDS:

    result = run_seed(
        seed
    )

    all_seed_results.append(
        result
    )


# ============================================================
# 15) BUILD LONG-FORM RESULTS TABLE
# ============================================================

rows = []


for result in all_seed_results:

    seed = result[
        "seed"
    ]


    for key, values in (
        result[
            "selection_results"
        ].items()
    ):

        rows.append({

            "seed":
                seed,

            "criterion":
                key,

            "selection_name":
                values[
                    "selection_name"
                ],

            "selected_epoch":
                values[
                    "selected_epoch"
                ],

            "cal_accuracy":
                values[
                    "reloaded_cal_accuracy"
                ],

            "cal_macro84":
                values[
                    "reloaded_cal_macro84"
                ],

            "cal_weighted_f1":
                values[
                    "reloaded_cal_weighted_f1"
                ],

            "test_accuracy":
                values[
                    "test_accuracy"
                ],

            "test_macro84":
                values[
                    "test_macro84"
                ],

            "test_weighted_f1":
                values[
                    "test_weighted_f1"
                ],
        })


long_df = pd.DataFrame(
    rows
).sort_values(
    [
        "criterion",
        "seed",
    ]
).reset_index(
    drop=True
)


long_path = (
    OUT_ROOT
    /
    "all_seed_checkpoint_selection_results.csv"
)


long_df.to_csv(
    long_path,
    index=False,
)


# ============================================================
# 16) MEAN ± SAMPLE SD
# ============================================================

summary_rows = []


criterion_order = [
    "macro_f1",
    "accuracy",
    "weighted_f1",
]


for criterion_name in criterion_order:

    group = (
        long_df[
            long_df[
                "criterion"
            ]
            ==
            criterion_name
        ]
        .copy()
    )


    record = {

        "criterion":
            criterion_name,

        "selection_name":
            group[
                "selection_name"
            ].iloc[0],

        "n_seeds":
            len(group),
    }


    for metric in [

        "selected_epoch",

        "cal_accuracy",

        "cal_macro84",

        "cal_weighted_f1",

        "test_accuracy",

        "test_macro84",

        "test_weighted_f1",
    ]:

        values = (
            group[
                metric
            ]
            .to_numpy(
                dtype=float
            )
        )


        mean = float(
            values.mean()
        )

        sd = float(
            values.std(
                ddof=1
            )
        )


        record[
            f"{metric}_mean"
        ] = mean

        record[
            f"{metric}_sd"
        ] = sd


        if metric != "selected_epoch":

            record[
                f"{metric}_paper"
            ] = (
                f"{mean * 100:.4f} "
                f"± "
                f"{sd * 100:.4f}%"
            )


    summary_rows.append(
        record
    )


summary_df = pd.DataFrame(
    summary_rows
)


summary_path = (
    OUT_ROOT
    /
    "checkpoint_selection_mean_sd.csv"
)


summary_df.to_csv(
    summary_path,
    index=False,
)


# ============================================================
# 17) MATCHED-SEED DIFFERENCES VS MACRO-F1 SELECTION
# ============================================================

macro_df = (
    long_df[
        long_df[
            "criterion"
        ]
        ==
        "macro_f1"
    ]
    .set_index(
        "seed"
    )
)


difference_rows = []


for alternative in [
    "accuracy",
    "weighted_f1",
]:

    alt_df = (
        long_df[
            long_df[
                "criterion"
            ]
            ==
            alternative
        ]
        .set_index(
            "seed"
        )
    )


    for metric in [

        "test_accuracy",

        "test_macro84",

        "test_weighted_f1",
    ]:

        differences = (
            alt_df[
                metric
            ]
            -
            macro_df[
                metric
            ]
        )


        difference_rows.append({

            "alternative_criterion":
                alternative,

            "reference_criterion":
                "macro_f1",

            "metric":
                metric,

            "seed42_delta_pp":
                float(
                    differences.loc[42]
                    *
                    100
                ),

            "seed43_delta_pp":
                float(
                    differences.loc[43]
                    *
                    100
                ),

            "seed44_delta_pp":
                float(
                    differences.loc[44]
                    *
                    100
                ),

            "mean_delta_pp":
                float(
                    differences.mean()
                    *
                    100
                ),

            "sd_delta_pp":
                float(
                    differences.std(
                        ddof=1
                    )
                    *
                    100
                ),
        })


difference_df = pd.DataFrame(
    difference_rows
)


difference_path = (
    OUT_ROOT
    /
    "matched_seed_differences_vs_macro.csv"
)


difference_df.to_csv(
    difference_path,
    index=False,
)


# ============================================================
# 18) DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    " MODEL-SELECTION SENSITIVITY COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nPER-SEED RESULTS"
)

display(
    long_df[
        [
            "seed",
            "selection_name",
            "selected_epoch",
            "cal_accuracy",
            "cal_macro84",
            "cal_weighted_f1",
            "test_accuracy",
            "test_macro84",
            "test_weighted_f1",
        ]
    ]
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "MEAN ± SAMPLE SD"
)

print(
    "=" * 100
)


display(
    summary_df[
        [
            "selection_name",
            "selected_epoch_mean",
            "selected_epoch_sd",
            "test_accuracy_paper",
            "test_macro84_paper",
            "test_weighted_f1_paper",
        ]
    ]
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "MATCHED-SEED DIFFERENCES VS MACRO-F1 SELECTION"
)

print(
    "=" * 100
)


display(
    difference_df
)


# ============================================================
# 19) FINAL 3-SEED PACKAGE
# ============================================================

bundle_path = Path(
    "/kaggle/working/"
    "MODEL_SELECTION_SENSITIVITY_3SEEDS.zip"
)


with zipfile.ZipFile(
    bundle_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as z:

    z.write(
        long_path,
        arcname=long_path.name,
    )

    z.write(
        summary_path,
        arcname=summary_path.name,
    )

    z.write(
        difference_path,
        arcname=difference_path.name,
    )


    for seed in SEEDS:

        seed_dir = (
            OUT_ROOT
            /
            f"seed_{seed}"
        )


        for path in seed_dir.iterdir():

            if path.is_file():

                z.write(
                    path,
                    arcname=(
                        f"seed_{seed}/"
                        f"{path.name}"
                    ),
                )


with zipfile.ZipFile(
    bundle_path,
    "r",
) as z:

    assert z.testzip() is None


print(
    "\nFinal package:"
)

print(
    bundle_path
)


print(
    "\nQA:"
)

print(
    " Morgan + RDKit = 2249 features"
)

print(
    " 84 Interaction IDs/labels preserved"
)

print(
    " Ordered Drug1→Drug2 preserved"
)

print(
    " Train graph = 268,244 edges"
)

print(
    " Same trajectory used for all selection criteria within each seed"
)

print(
    " Calibration-only checkpoint selection"
)

print(
    " Best Macro-F1 checkpoint saved"
)

print(
    " Best Accuracy checkpoint saved"
)

print(
    " Best Weighted-F1 checkpoint saved"
)

print(
    " Test touched only after training/model selection"
)

print(
    " 3 seeds = 42, 43, 44"
)

print(
    " Mean ± sample SD generated"
)

print(
    " Matched-seed differences generated"
)

print(
    " Old feature-ablation artifacts untouched"
)

print(
    " ZIP integrity passed"
)